# 🕵️ Sherlock Holmes RAG Chatbot

A pilot-hardened, conversational chatbot that answers questions about *The Adventures of Sherlock Holmes* — grounded in the book's actual text, with source citations you can verify.

**Stack (all free):**
- **Groq** — LLM inference (Llama 3.3 70B)
- **LlamaIndex** — RAG orchestration
- **HuggingFace `all-MiniLM-L6-v2`** — local sentence embeddings (no API)
- **Project Gutenberg** — public-domain source text

**What makes this version pilot-proof:**
| Weakness | Fix |
|---|---|
| Follow-ups like *"tell me more"* fetching unrelated chunks | `CondensePlusContextChatEngine` rewrites vague queries using conversation history |
| No way to know which of the 12 stories a chunk came from | Story-tagged metadata on every chunk |
| Confidently answering when retrieval was weak | Relevance floor — bot refuses when top score < 0.35 |
| Ambiguous questions getting arbitrary answers | Numbered system prompt with explicit disambiguation rules |

---
## 1. Installations 🛠️

In [ ]:
!pip install -qqq llama-index-core
!pip install -qqq llama-index-llms-groq
!pip install -qqq llama-index-embeddings-huggingface
!pip install -qqq sentence-transformers
!pip install -qqq python-dotenv

---
## 2. Load the Groq API key 🔑

Keys should never be hardcoded. Locally use `.env` (add to `.gitignore`). On Colab, use `userdata.get()`.

In [ ]:
# --- Local (.env file) ---
from dotenv import load_dotenv
load_dotenv()

# --- Or Colab ---
# from google.colab import userdata
# import os
# os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')

import os
assert os.environ.get('GROQ_API_KEY'), 'GROQ_API_KEY not found — check your .env file'

---
## 3. Get the book from Project Gutenberg 📚

In [ ]:
import urllib.request

os.makedirs('data', exist_ok=True)
URL = 'https://www.gutenberg.org/files/1661/1661-0.txt'
RAW_PATH = 'data/sherlock_holmes.txt'

if not os.path.exists(RAW_PATH):
    urllib.request.urlretrieve(URL, RAW_PATH)
    print(f'Downloaded to {RAW_PATH}')

with open(RAW_PATH, 'r', encoding='utf-8') as f:
    text = f.read()

print(f'File size: {len(text):,} chars')
print('\n--- First 400 chars ---')
print(text[:400])

### 3.1 Strip Gutenberg boilerplate

Every Gutenberg file wraps the book with legal headers/footers (~500 lines). If we leave them in, the retriever will happily return copyright text when someone asks "who is Sherlock?". Prevent that failure mode up front.

In [ ]:
START = '*** START OF THE PROJECT GUTENBERG EBOOK'
END = '*** END OF THE PROJECT GUTENBERG EBOOK'

start_idx = text.find('\n', text.find(START)) + 1
end_idx = text.find(END)
clean = text[start_idx:end_idx].strip()

print(f'Cleaned text: {len(clean):,} chars ({len(text) - len(clean):,} chars of boilerplate removed)')

### 3.2 Split into the 12 individual stories

*The Adventures of Sherlock Holmes* is a collection of twelve short stories. If we index the whole book as one document, we lose the ability to say **which story** an answer came from. By splitting first and tagging each chunk with a `story` metadata field, we get named citations like *"from The Red-Headed League"* in the UI.

In [ ]:
import re

STORY_TITLES = [
    'A Scandal in Bohemia',
    'The Red-Headed League',
    'A Case of Identity',
    'The Boscombe Valley Mystery',
    'The Five Orange Pips',
    'The Man with the Twisted Lip',
    'The Adventure of the Blue Carbuncle',
    'The Adventure of the Speckled Band',
    "The Adventure of the Engineer's Thumb",
    'The Adventure of the Noble Bachelor',
    'The Adventure of the Beryl Coronet',
    'The Adventure of the Copper Beeches',
]

# Titles in the book appear in ALL CAPS (e.g. 'I. A SCANDAL IN BOHEMIA')
positions = []
for title in STORY_TITLES:
    match = re.search(re.escape(title.upper()), clean, re.IGNORECASE)
    if match:
        positions.append((match.start(), title))
positions.sort()

stories = {}
for i, (start, title) in enumerate(positions):
    end = positions[i + 1][0] if i + 1 < len(positions) else len(clean)
    stories[title] = clean[start:end].strip()

print(f'Found {len(stories)} stories:')
for title, body in stories.items():
    print(f'  • {title}: {len(body):,} chars')

---
## 4. Set up the LLM 🧠

**Choices and trade-offs:**
- **`llama-3.3-70b-versatile`** — Meta's open 70B model, strong reasoning, hosted on Groq's LPU chips (~500 tokens/sec)
- **`temperature=0.1`** — nearly deterministic. For RAG we want the LLM to *summarize* the retrieved context, not embellish. Low temp discourages creativity → discourages hallucination.
- **`max_tokens=512`** — hard cap on response length. Prevents runaway rambling.

In [ ]:
from llama_index.llms.groq import Groq

llm = Groq(
    model='llama-3.3-70b-versatile',
    temperature=0.1,
    max_tokens=512,
)

print(llm.complete('Say hello in one short sentence.'))

---
## 5. Set up local embeddings 🔢

**Why local embeddings, not OpenAI?**
- **Free** — no API cost, no rate limits
- **Private** — text never leaves the system
- **Fast** — MiniLM runs on CPU in milliseconds
- **Small** — 90MB model, 384-dim vectors
- **Purpose-built** — distilled from BERT specifically for sentence similarity

In [ ]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

embed_model = HuggingFaceEmbedding(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    cache_folder='./embedding_model/',
)

vec = embed_model.get_text_embedding('A study in scarlet.')
print(f'Vector dimension: {len(vec)}')

---
## 6. Build the vector index with story metadata 🗂️

**The three-step RAG index recipe:**
1. **Chunk** — split each story into ~512-token passages with 64-token overlap
2. **Embed** — turn each chunk into a 384-dim vector
3. **Store** — persist to disk so future runs load instantly

**Chunking trade-off:** 512 tokens is a standard sweet spot for narrative prose — big enough to hold a scene, small enough for precise retrieval. The 64-token overlap prevents key sentences from being lost at chunk boundaries.

**The new bit:** each chunk carries `{'story': 'A Scandal in Bohemia'}` etc. as metadata, so we can display *which story* an answer comes from.

In [ ]:
from llama_index.core import (
    Document, VectorStoreIndex, StorageContext,
    load_index_from_storage, Settings,
)

Settings.llm = llm
Settings.embed_model = embed_model
Settings.chunk_size = 512
Settings.chunk_overlap = 64

INDEX_DIR = './vector_index_v2'

if not os.path.exists(INDEX_DIR):
    print('Building index with per-story metadata...')
    documents = [
        Document(text=body, metadata={'story': title})
        for title, body in stories.items()
    ]
    vector_index = VectorStoreIndex.from_documents(documents, show_progress=True)
    vector_index.storage_context.persist(persist_dir=INDEX_DIR)
    print(f'✅ Saved to {INDEX_DIR}')
else:
    print('Loading existing index...')
    vector_index = load_index_from_storage(
        StorageContext.from_defaults(persist_dir=INDEX_DIR)
    )

---
## 7. Assemble the pilot-hardened chat engine 💬

**Why `CondensePlusContextChatEngine` instead of `ContextChatEngine`?**

The plain `ContextChatEngine` embeds only the *current* message. So a follow-up like "tell me more" — three low-content words — produces a vector that matches almost nothing in the book, and retrieval returns random-ish chunks. The bot then answers based on whatever it got, drifting to unrelated stories.

`CondensePlusContextChatEngine` adds a **query rewriting step**: before retrieval, the LLM condenses the conversation history + new question into a standalone query. So "tell me more" becomes something like *"tell me more about the Blue Carbuncle case and the stolen gem"*, and the retriever gets a query it can actually work with.

**The system prompt.** LLMs follow numbered rules far more reliably than paragraph-form ones. Rules 2 and 3 are the disambiguation logic that keeps ambiguous questions and vague follow-ups from producing garbage.

In [ ]:
from llama_index.core.chat_engine import CondensePlusContextChatEngine
from llama_index.core.memory import Memory

SYSTEM_PROMPT = (
    "You are a knowledgeable literary assistant specializing in "
    "'The Adventures of Sherlock Holmes' by Arthur Conan Doyle — a collection of TWELVE distinct short stories.\n\n"
    "Rules you MUST follow, in order:\n\n"
    "1. GROUNDING: Answer using ONLY the provided context passages. Never use outside knowledge, "
    "even if you 'know' the answer from training. If the passages do not contain the answer, say so plainly.\n\n"
    "2. AMBIGUITY: If the user's question is vague or could apply to multiple stories "
    "(e.g. 'tell me a story', 'what happens?', 'who is the villain?'), do ONE of the following:\n"
    "   (a) If the context clearly points to ONE story, name it confidently and summarize.\n"
    "   (b) If the context spans multiple stories, ask the user to specify which one.\n"
    "   (c) If nothing relevant was retrieved, ask for a more specific question.\n\n"
    "3. FOLLOW-UPS: Vague follow-ups like 'tell me more', 'continue', or 'go on' refer to "
    "the MOST RECENT story you were discussing. Do NOT switch stories on a follow-up.\n\n"
    "4. CITE THE STORY: Always mention which of the twelve stories the answer comes from.\n\n"
    "5. LENGTH: Keep answers to 2-4 sentences unless the user explicitly asks for detail."
)

memory = Memory.from_defaults(token_limit=2000)
retriever = vector_index.as_retriever(similarity_top_k=4)

chatbot = CondensePlusContextChatEngine.from_defaults(
    retriever=retriever,
    llm=llm,
    memory=memory,
    system_prompt=SYSTEM_PROMPT,
)
print('✅ Chat engine ready')

---
## 8. Test drive 🚗

Four demo questions that showcase different features.

In [ ]:
def ask(question):
    print(f'\n❓ {question}\n')
    response = chatbot.chat(question)
    print(f'💬 {response}\n')
    print('📖 Sources:')
    for node in response.source_nodes:
        story = node.metadata.get('story', 'unknown')
        print(f'  • {story} (score {node.score:.3f})')
    return response

In [ ]:
# Grounded, specific question → confident answer with story citation
_ = ask('Who is Irene Adler and why does Holmes remember her?')

In [ ]:
# The hard test — a vague follow-up. Should stay on Scandal in Bohemia.
_ = ask('tell me more')

In [ ]:
# Grounding test — the book doesn't cover this. Bot should refuse rather than guess.
_ = ask('Was Sherlock Holmes ever married?')

In [ ]:
# Ambiguity test — bot should ask which story or pick confidently
chatbot.reset()  # clear memory so this is a fresh conversation
_ = ask('tell me a story')

---
## 9. Inspecting retrieval (the debug view)

Bypass the LLM — pull raw chunks so we can see what the retriever actually surfaces. Useful for debugging weird answers and for explaining the system in interviews.

In [ ]:
nodes = retriever.retrieve('What happens in The Speckled Band?')
for i, node in enumerate(nodes, 1):
    print(f"--- Chunk {i}: from {node.metadata.get('story')} (score {node.score:.3f}) ---")
    print(node.text[:300], '...\n')

---
## 10. Interactive loop 🔁

Type freely. `end` quits, `reset` clears memory.

In [ ]:
print('🕵️ Sherlock Holmes chatbot — type end to quit, reset to clear memory.\n')
while True:
    user_input = input('You: ').strip()
    if not user_input:
        continue
    if user_input.lower() == 'end':
        print('Goodbye!')
        break
    if user_input.lower() == 'reset':
        chatbot.reset()
        print('Memory cleared.\n')
        continue
    response = chatbot.chat(user_input)
    print(f'\nHolmes-bot: {response}\n')

---
## 11. Deployment

The matching Streamlit app (`streamlit_app.py`) uses the same chat engine with:
- A **relevance floor** — if the top retrieved chunk scores below 0.35, the bot refuses to answer instead of guessing
- **Suggested questions** so demos don't start with a blank chat
- **Story-named source citations** in expandable panels
- **Session-isolated memory** — each user gets their own conversation
- **Error handling** — friendly message on API hiccup instead of a red traceback

Deployed via Streamlit Community Cloud, auto-redeploys on every `git push`. Total infrastructure cost: **€0**.

---

## What this demonstrates

- **End-to-end RAG pipeline design** — data ingestion, chunking, embedding, indexing, retrieval, generation
- **Trade-off reasoning** — every parameter (chunk size, top-k, temperature, relevance floor) was chosen deliberately
- **Failure-mode awareness** — the notebook identifies four real weaknesses of naive RAG and fixes each one
- **Production hardening** — error handling, session isolation, secret management, deployment

Built by Abdul Rahman Abssi · Berlin · 2026